# RiskWise-AI Claims Data Exploration

This notebook explores the local `data/insurance_claims.csv` dataset for fraud, claim amount, incident, policy, vehicle, and reporting patterns. The CSV is intentionally ignored by Git, so run this notebook after placing the dataset in `data/insurance_claims.csv`.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache" / "matplotlib"))

import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

from backend.app.data.data_loader import load_claims_data  # noqa: E402

ASSET_DIR = PROJECT_ROOT / "docs" / "assets" / "eda"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
claims = load_claims_data()

print(f"Rows: {len(claims):,}")
print(f"Columns: {len(claims.columns):,}")
claims.head()

In [ ]:
claims.describe(include="all").transpose().head(20)

In [ ]:
def save_bar(
    series: pd.Series, title: str, ylabel: str, filename: str, *, rotation: int = 0
) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    series.plot(kind="bar", ax=ax, color="#2f6f9f")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotation)
    fig.tight_layout()
    fig.savefig(ASSET_DIR / filename, dpi=160)
    plt.show()


def save_hist(series: pd.Series, title: str, xlabel: str, filename: str) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(series.dropna(), bins=30, color="#2f6f9f", edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Claim count")
    fig.tight_layout()
    fig.savefig(ASSET_DIR / filename, dpi=160)
    plt.show()


def save_scatter(
    x: pd.Series, y: pd.Series, title: str, xlabel: str, ylabel: str, filename: str
) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(x, y, alpha=0.55, color="#2f6f9f", edgecolor="none")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    fig.savefig(ASSET_DIR / filename, dpi=160)
    plt.show()

## Fraud Distribution

In [ ]:
fraud_counts = claims["fraud_reported"].map({0: "Not reported", 1: "Reported"}).value_counts()
save_bar(fraud_counts, "Fraud Report Distribution", "Claims", "fraud_distribution.png")
fraud_counts

## Claim Amount Distribution

In [ ]:
save_hist(
    claims["total_claim_amount"],
    "Total Claim Amount Distribution",
    "Total claim amount",
    "claim_amount_distribution.png",
)

## Incident Severity Vs Fraud

In [ ]:
severity_fraud = (
    claims.groupby("incident_severity")["fraud_reported"].mean().sort_values(ascending=False)
)
save_bar(
    severity_fraud,
    "Fraud Rate By Incident Severity",
    "Fraud rate",
    "incident_severity_vs_fraud.png",
    rotation=25,
)
severity_fraud

## Policy Premium Vs Claim Amount

In [ ]:
save_scatter(
    claims["policy_annual_premium"],
    claims["total_claim_amount"],
    "Policy Annual Premium Vs Total Claim Amount",
    "Policy annual premium",
    "Total claim amount",
    "policy_premium_vs_claim_amount.png",
)

## Auto Year Vs Claim Amount

In [ ]:
auto_year_claim = claims.groupby("auto_year")["total_claim_amount"].mean().sort_index()
fig, ax = plt.subplots(figsize=(8, 5))
auto_year_claim.plot(ax=ax, marker="o", color="#2f6f9f")
ax.set_title("Average Claim Amount By Auto Year")
ax.set_xlabel("Auto year")
ax.set_ylabel("Average total claim amount")
fig.tight_layout()
fig.savefig(ASSET_DIR / "auto_year_vs_claim_amount.png", dpi=160)
plt.show()
auto_year_claim.tail()

## State-Wise Claim Patterns

In [ ]:
state_claims = (
    claims.groupby("policy_state")["total_claim_amount"].mean().sort_values(ascending=False)
)
save_bar(
    state_claims,
    "Average Claim Amount By Policy State",
    "Average total claim amount",
    "state_claim_patterns.png",
)
state_claims

## Bodily Injuries Vs Claim Amount

In [ ]:
injury_claims = claims.groupby("bodily_injuries")["total_claim_amount"].mean().sort_index()
save_bar(
    injury_claims,
    "Average Claim Amount By Bodily Injuries",
    "Average total claim amount",
    "bodily_injuries_vs_claim_amount.png",
)
injury_claims

## Witnesses Vs Fraud

In [ ]:
witness_fraud = claims.groupby("witnesses")["fraud_reported"].mean().sort_index()
save_bar(witness_fraud, "Fraud Rate By Witness Count", "Fraud rate", "witnesses_vs_fraud.png")
witness_fraud

## Police Report Availability Vs Fraud

In [ ]:
police_report_fraud = (
    claims.groupby("police_report_available")["fraud_reported"].mean().sort_values(ascending=False)
)
save_bar(
    police_report_fraud,
    "Fraud Rate By Police Report Availability",
    "Fraud rate",
    "police_report_availability_vs_fraud.png",
)
police_report_fraud